In [ ]:
from statsmodels.stats.multitest import multipletests

import pandas as pd
from scipy import stats
import numpy as np
import anndata as ad

adata = ad.read_h5ad("/home/miguel-agromayor-otero/Escritorio/TFM_datos/mofa_adata_30f.h5ad")
factors_matrix = adata.X
num_factors = factors_matrix.shape[1]

for metadata_col in ['plate', 'drug', 'moa-fine', 'concentration']:
    results = []
    print(f"\nAnalysing against '{metadata_col}'")

    for i in range(num_factors):
        df_temp = pd.DataFrame({
            'value': factors_matrix[:, i],
            'group': adata.obs[metadata_col].values
        }).dropna()

        groups = [g['value'].values for _, g in df_temp.groupby('group', observed=True)]
        if len(groups) > 1:
            f_stat, p_val = stats.f_oneway(*groups)
        else:
            f_stat, p_val = 0.0, 1.0
        
        

        results.append({
            'Factor': f"Factor{i+1}",
            'F_Stat': round(f_stat, 4),
            'P_Value': p_val,
        
        })
       

    df_res = pd.DataFrame(results)
    
    # Corrección FDR
    df_res['P_adj'] = multipletests(df_res['P_Value'], method='fdr_bh')[1]
    df_res['Veredicto'] = df_res['P_adj'].apply(
        lambda x: 'Significativo' if x < 0.05 else 'No significativo'
    )

    print(df_res.to_string(index=False))
    
    out = f'/home/ulc/co/mao/drug_clustering/TFM_mao/resultados/technical_batch/anova_{metadata_col.replace("-","_")}.xlsx'
    df_res.to_excel(out, index=False)
    print(f"Save in {out}")